<a href="https://colab.research.google.com/github/computer0796579/Angular/blob/main/Database_practice_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
with open('/content/policy-engine-eval-harness-design.md', 'r') as f:
    design_doc = f.read()

print(design_doc)

# Policy Engine Eval Harness — Design Sketch

## Goal

Turn the Policy Engine's decision logic into something you can regression-test the same way you'd test any other critical function: given a known input, assert the expected output, run it on every change, fail the build if it drifts.

This deliberately scopes to **policy-engine evals** (deterministic allow/deny/escalate correctness), not agent-behavior evals (fuzzy LLM-as-judge scoring of full agent traces). That's a separate, harder harness — worth doing later, once this one is solid.

---

## 1. Golden Dataset Schema

Store as a table in the existing SQLite DB (or a separate `evals.db` if you want write isolation from production data — recommended, since eval runs will INSERT frequently and you don't want that anywhere near the audit log).

```sql
CREATE TABLE eval_cases (
    id TEXT PRIMARY KEY,              -- e.g. "case-0042" or a slug
    description TEXT NOT NULL,        -- human-readable: what this case tests
    category 

In [2]:
import sqlite3

def setup_database(db_path='evals.db'):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 1. Create eval_cases table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS eval_cases (
        id TEXT PRIMARY KEY,
        description TEXT NOT NULL,
        category TEXT NOT NULL,
        source TEXT NOT NULL,
        source_ref TEXT,
        agent_id TEXT NOT NULL,
        action_type TEXT NOT NULL,
        request_payload TEXT NOT NULL,
        context_payload TEXT,
        expected_decision TEXT NOT NULL,
        expected_reason_code TEXT,
        expected_policy_version INTEGER
    )
    ''')

    # 2. Create eval_runs table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS eval_runs (
        id TEXT PRIMARY KEY,
        run_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        policy_version INTEGER NOT NULL,
        total_cases INTEGER,
        passed_cases INTEGER,
        failed_cases INTEGER
    )
    ''')

    # 3. Create eval_results table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS eval_results (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id TEXT NOT NULL,
        case_id TEXT NOT NULL,
        passed BOOLEAN NOT NULL,
        actual_decision TEXT,
        actual_reason_code TEXT,
        error_message TEXT,
        FOREIGN KEY (run_id) REFERENCES eval_runs(id),
        FOREIGN KEY (case_id) REFERENCES eval_cases(id)
    )
    ''')

    conn.commit()
    conn.close()
    print(f'Database initialized at {db_path}')

setup_database()

Database initialized at evals.db


In [3]:
import sqlite3
import json

def seed_eval_cases(db_path='evals.db'):
    cases = [
        (
            'case-001',
            'Allow Gmail send within normal limits',
            'connector_scope',
            'hand_written',
            None,
            'agent-alpha',
            'gmail.send',
            json.dumps({'recipient': 'user@example.com', 'subject': 'Hello'}),
            json.dumps({'rate_limit_count': 5, 'trust_tier': 1}),
            'allow',
            None,
            1
        ),
        (
            'case-002',
            'Deny Gmail send when rate limit exceeded',
            'rate_limit',
            'hand_written',
            None,
            'agent-alpha',
            'gmail.send',
            json.dumps({'recipient': 'user@example.com'}),
            json.dumps({'rate_limit_count': 101, 'trust_tier': 1}),
            'deny',
            'RATE_LIMIT_EXCEEDED',
            1
        ),
        (
            'case-003',
            'Escalate sensitive Jira transition',
            'escalation',
            'hand_written',
            None,
            'agent-beta',
            'jira.transition',
            json.dumps({'issue_id': 'PROJ-123', 'status': 'Closed'}),
            json.dumps({'requires_approval': True}),
            'escalate',
            'APPROVAL_REQUIRED',
            1
        )
    ]

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.executemany('''
        INSERT OR REPLACE INTO eval_cases (
            id, description, category, source, source_ref, agent_id,
            action_type, request_payload, context_payload,
            expected_decision, expected_reason_code, expected_policy_version
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', cases)
    conn.commit()
    conn.close()
    print(f'Successfully seeded {len(cases)} evaluation cases.')

seed_eval_cases()

Successfully seeded 3 evaluation cases.


In [4]:
import sqlite3
import json
import uuid
from datetime import datetime

# Mock Policy Engine for testing the runner
def mock_policy_engine(agent_id, action_type, request_payload, context_payload):
    request = json.loads(request_payload)
    context = json.loads(context_payload) if context_payload else {}

    # Example logic matching our seeded cases
    if action_type == 'gmail.send':
        if context.get('rate_limit_count', 0) > 100:
            return 'deny', 'RATE_LIMIT_EXCEEDED'
        return 'allow', None

    if action_type == 'jira.transition':
        if context.get('requires_approval'):
            return 'escalate', 'APPROVAL_REQUIRED'

    return 'deny', 'UNKNOWN_ACTION'

def run_eval_suite(db_path='evals.db', policy_version=1):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    # 1. Fetch cases
    cases = cursor.execute('SELECT * FROM eval_cases').fetchall()
    run_id = str(uuid.uuid4())[:8]

    results = []
    passed_count = 0

    # 2. Execute each case
    for case in cases:
        actual_decision, actual_reason = mock_policy_engine(
            case['agent_id'],
            case['action_type'],
            case['request_payload'],
            case['context_payload']
        )

        passed = (actual_decision == case['expected_decision']) and \
                 (actual_reason == case['expected_reason_code'])

        if passed: passed_count += 1

        results.append((
            run_id,
            case['id'],
            passed,
            actual_decision,
            actual_reason
        ))

    # 3. Record the run
    cursor.execute('''
        INSERT INTO eval_runs (id, policy_version, total_cases, passed_cases, failed_cases)
        VALUES (?, ?, ?, ?, ?)
    ''', (run_id, policy_version, len(cases), passed_count, len(cases) - passed_count))

    # 4. Record results
    cursor.executemany('''
        INSERT INTO eval_results (run_id, case_id, passed, actual_decision, actual_reason_code)
        VALUES (?, ?, ?, ?, ?)
    ''', results)

    conn.commit()
    conn.close()

    print(f'Eval Run {run_id} completed: {passed_count}/{len(cases)} passed.')
    return run_id

# Execute a test run
run_eval_suite()

Eval Run 98a405e1 completed: 3/3 passed.


'98a405e1'

In [5]:
from fastapi import FastAPI, HTTPException
import sqlite3

app = FastAPI()
DB_PATH = 'evals.db'
CURRENT_POLICY_VERSION = 1

@app.post("/evals/run")
async def trigger_eval_run():
    # Reusing the runner function defined in the previous cell
    run_id = run_eval_suite(db_path=DB_PATH, policy_version=CURRENT_POLICY_VERSION)

    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    run = conn.execute("SELECT * FROM eval_runs WHERE id = ?", (run_id,)).fetchone()
    conn.close()
    return dict(run)

@app.get("/evals/runs")
async def list_eval_runs(limit: int = 20):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    runs = conn.execute("SELECT * FROM eval_runs ORDER BY run_at DESC LIMIT ?", (limit,)).fetchall()
    conn.close()
    return [dict(r) for r in runs]

@app.get("/evals/runs/{run_id}/results")
async def get_eval_run_results(run_id: str):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    results = conn.execute(
        "SELECT * FROM eval_results WHERE run_id = ? ORDER BY passed ASC",
        (run_id,)
    ).fetchall()
    conn.close()

    if not results:
        raise HTTPException(status_code=404, detail="Run results not found")

    return [dict(r) for r in results]

print("FastAPI endpoints defined successfully.")

FastAPI endpoints defined successfully.


In [6]:
def evaluate_policy(agent_id: str, action_type: str, request_payload: dict, context_payload: dict = None):
    """
    Mock policy evaluation logic.
    In a real system, this would query the active policy rules.
    """
    # Example logic based on common security patterns
    if action_type == 'gmail.send':
        count = context_payload.get('rate_limit_count', 0) if context_payload else 0
        if count > 100:
            return 'deny', 'RATE_LIMIT_EXCEEDED'
        return 'allow', None

    if action_type == 'jira.transition':
        if context_payload and context_payload.get('requires_approval'):
            return 'escalate', 'APPROVAL_REQUIRED'
        return 'allow', None

    # Default fallback
    return 'deny', 'UNAUTHORIZED_ACTION'

In [7]:
def run_eval_suite_v2(db_path='evals.db', policy_version=1):
    """Updated runner using the external evaluate_policy function."""
    import sqlite3
    import json
    import uuid

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    cases = cursor.execute('SELECT * FROM eval_cases').fetchall()
    run_id = str(uuid.uuid4())[:8]
    results = []
    passed_count = 0

    for case in cases:
        # Parse payloads for the function
        req = json.loads(case['request_payload'])
        ctx = json.loads(case['context_payload']) if case['context_payload'] else {}

        # Call our mock engine
        actual_decision, actual_reason = evaluate_policy(
            case['agent_id'],
            case['action_type'],
            req,
            ctx
        )

        passed = (actual_decision == case['expected_decision']) and \
                 (actual_reason == case['expected_reason_code'])

        if passed: passed_count += 1

        results.append((run_id, case['id'], passed, actual_decision, actual_reason))

    # Log run summary
    cursor.execute('''
        INSERT INTO eval_runs (id, policy_version, total_cases, passed_cases, failed_cases)
        VALUES (?, ?, ?, ?, ?)
    ''', (run_id, policy_version, len(cases), passed_count, len(cases) - passed_count))

    # Log individual results
    cursor.executemany('''
        INSERT INTO eval_results (run_id, case_id, passed, actual_decision, actual_reason_code)
        VALUES (?, ?, ?, ?, ?)
    ''', results)

    conn.commit()
    conn.close()

    print(f'Eval Run {run_id} (V2) completed: {passed_count}/{len(cases)} passed.')
    return run_id

# Run the updated suite
run_eval_suite_v2()

Eval Run 44fb93ce (V2) completed: 3/3 passed.


'44fb93ce'